# Run the real pipeline

Unlike `build_demo_db.ipynb` (hand-authored, illustrative data, no network
calls), this notebook calls `protein_selector.pipeline.run_pipeline` for
real: real RCSB hard-filters search, real simulability/composition checks,
real ligand CCD/SMILES lookup, real RDKit (and Meeko, if installed)
parameterizability, real Europe PMC literature counts, and a real AlphaFold
DB lookup for ex02 -- every number below comes from an actual API response,
nothing is invented.

**Left off on purpose** (see `pipeline.py`'s own docstring for why):
- `run_ex03` (real OpenMM MD) -- needs the conda-only `environment-validation.yml` env,
  not installed in this base venv.
- `run_pocket_detection` (real fpocket) -- needs a local `fpocket` binary.
- ex04 (real Vina docking + PLIP) isn't wired into the pipeline at all yet --
  no receptor-prep/pocket-center-extraction code exists to auto-run it.

So ex03/ex04 will show `"not_run"` below -- that's accurate, not a bug:
this notebook genuinely hasn't run those checks, unlike the demo notebook's
hardcoded pass/fail rows for them.

`max_candidates` is kept small (a real Meeko 3D-embedding step, if
installed, can be slow for some real bound ligands) -- raise it once you've
confirmed a run completes in reasonable time for your machine.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from protein_selector.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Separate db from build_demo_db.ipynb's illustrative one, so the two never mix.
DB_PATH = Path("cache/protein_selector_real.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
CSV_PATH = Path("report_real.csv")

In [ ]:
rows = run_pipeline(
    db_path=DB_PATH,
    max_candidates=3,
    run_ex02=True,
    run_ex03=False,
    run_pocket_detection=False,
    report_csv_path=CSV_PATH,
)
len(rows)

## The real joined report

In [ ]:
from protein_selector.core.report import rows_to_dataframe

report_df = rows_to_dataframe(rows)
report_df

## What's real here, spelled out

- `title`/`organism`/`n_residues`/`resolution`/... -- real RCSB entry metadata.
- `ligand_parameterizable` -- a real RDKit sanitization result (and real Meeko
  3D-embed + PDBQT-write result, if the `validate` extra is installed).
- `litref_count` -- a real Europe PMC hit count for this exact PDB ID.
- `ex02_status`/`ex02_predicted_difficulty` -- a real AlphaFold DB lookup: if
  the candidate's UniProt accession has a modeled entry, this reflects its
  actual published confidence fractions; if not, `ex02_status` is `"fail"`
  with `FailureMode.COMPLETENESS`, not guessed.

In [ ]:
report_df[["pdb_id", "uniprot_id", "ligand_ccd", "ligand_parameterizable", "litref_count", "ex02_status", "ex02_predicted_difficulty", "ex02_failure_mode"]]